In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy as dc
import math
import pickle
import sys
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F

from Modules.LmylFun import LSTM, BLSTM, SimpRNN, NLin, DLin, Lin
from Modules.LmylFun import VMD
from Modules.LmylFun import PostVMD

In [ ]:
# @title Loading Data & Preparing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_steps = 6

# List Of Training And Test Data Set
trn_li = []
tst_li = []
scale_li = []

# Univariate Data List
uni_list = ['banjarbaru_pr',
            'batam_pr',
            'jakarta_pr',
            'jakarta-selatan_pr',
            'jambi_pr',
            'malang_pr',
            'medan_pr',
            'samarinda_pr',
            'semarang_pr']

for fileName in uni_list:
    scaler = StandardScaler()
    df = pd.read_csv("/DATASET/Univariate/"+fileName+".csv")
    df = df[['Unnamed: 0','pm25']]
    df.drop(df.tail(1).index,inplace=True)

    df_tr = df[:int(len(df)*(90/100))]
    df_tr = pd.DataFrame(scaler.fit_transform(df_tr[['pm25']]), columns=['pm25'])

    df_ts = df[-(int(len(df)*(10/100))):]
    df_ts = pd.DataFrame(scaler.transform(df_ts[['pm25']]), columns=['pm25'])

    trn_li.append(df_tr)
    tst_li.append(df_ts)
    scale_li.append(scaler)

# M4 WEEKLY TRAIN DATA
scaler = StandardScaler()
df_tr = pd.read_csv("/DATASET/Multivariate/m4-week-train.csv")
df_tr['V1'] = pd.to_datetime(df_tr['V1'], format='%Y-%m-%d')
df_tr = df_tr.iloc[:-5]
df_tr = df_tr.set_index('V1')
df_tr = pd.DataFrame(scaler.fit_transform(df_tr[['V2']]), columns=['V2'])

trn_li.append(df_tr)

# M4 WEEKLY TEST DATA
df_ts = pd.read_csv("/DATASET/Multivariate/m4-week-test.csv")
df_ts['V1'] = pd.to_datetime(df_ts['V1'], format='%Y-%m-%d')
df_ts = df_ts[:-5]
df_ts = df_ts.set_index('V1')
df_ts = pd.DataFrame(scaler.transform(df_ts[['V2']]), columns=['V2'])

tst_li.append(df_ts)
scale_li.append(scaler)

# SINGAPORE PM10 DATA
scaler = StandardScaler()
df = pd.read_csv("/DATASET/Multivariate/central_singapore.csv")
df.drop(df.tail(1).index,inplace=True)
df_tr = df[:int(len(df)*(90/100))]
df_tr = pd.DataFrame(scaler.fit_transform(df_tr[['pm10']]), columns=['pm10'])

df_ts = df[-(int(len(df)*(10/100))):]
df_ts = pd.DataFrame(scaler.transform(df_ts[['pm10']]), columns=['pm10'])
trn_li.append(df_tr)
tst_li.append(df_ts)
scale_li.append(scaler)

# WIND TURBINE ACTIVEPOWER DATA
scaler = StandardScaler()
df = pd.read_csv("/DATASET/Multivariate/Turbine_neo.csv")
df = df[:-10]
df_tr = df[:int(len(df)*(80/100))]
df_tr = pd.DataFrame(scaler.fit_transform(df_tr[['LV ActivePower (kW)']]), columns=['LV ActivePower (kW)'])

df_ts = df[-(int(len(df)*(20/100))):]
df_ts = pd.DataFrame(scaler.transform(df_ts[['LV ActivePower (kW)']]), columns=['LV ActivePower (kW)'])
trn_li.append(df_tr)
tst_li.append(df_ts)
scale_li.append(scaler)

# ETTm2 HUFL DATA
scaler = StandardScaler()
df = pd.read_csv("/DATASET/Multivariate/ETTm2_neo.csv")
df = df[:-59690]
df_tr = df[:int(len(df)*(80/100))]
df_tr = pd.DataFrame(scaler.fit_transform(df_tr[['HUFL']]), columns=['HUFL'])

df_ts = df[-(int(len(df)*(20/100))):]
df_ts = pd.DataFrame(scaler.transform(df_ts[['HUFL']]), columns=['HUFL'])
trn_li.append(df_tr)
tst_li.append(df_ts)
scale_li.append(scaler)

# The Train & Test List Data by Index
# 1. Banjarbaru (PM25)
# 2. Batam (PM25)
# 3. Jakarta (PM25)
# 4. Jakarta-Selatan (PM25)
# 5. Jambi (PM25)
# 6. Malang (PM25)
# 7. Medan (PM25)
# 8. Samarinda (PM25)
# 9. Semarang (PM25)
# 10. M4 Weekly (V1)
# 11. Singapore (PM10)
# 12. WindTurbine (LV ActivePower (kW))
# 13. ETTm2 (HUFL)

In [5]:
# @title The VMD Function
def TheVMD(tr, ts, colName, alpha, tau, K, tol, bch, n_step):
    u, u_hat, omega = VMD(tr[colName], alpha, tau, K, 0, 0, tol)
    ut, ut_hat = PostVMD(ts[colName], alpha, tau, K, 0, tol, omega[-1])

    # Preparing Data
    Xs = tr.copy()
    Xts = ts.copy()

    liXs = []
    liXts = []

    for i in range(len(u)):
        Xs[colName] = u[i]
        Xts[colName] = ut[i]
        liXs.append(preparing(Xs, n_step, colName))
        liXts.append(preparing(Xts, n_step, colName))

    for i in range(len(liXs)):
        liXs[i] = liXs[i].drop(index=liXs[i].index[:n_step])
        liXts[i] = liXts[i].drop(index=liXts[i].index[:n_step])

    # Data Splitting
    ys = []
    yts = []

    for a in range(len(liXs)):
        ys.append(liXs[a].pop(colName))
        yts.append(liXts[a].pop(colName))

    # Shifting To Numpy
    liXs_np = []
    liXts_np = []
    ys_np = []
    yts_np = []

    for i in range(len(liXs)):
        liXs_np.append(liXs[i].to_numpy())
        liXts_np.append(liXts[i].to_numpy())
        ys_np.append(ys[i].to_numpy())
        yts_np.append(yts[i].to_numpy())

    # Flipping Train Columns For RNN-Based Algorithm
    for i in range(len(liXs)):
        liXs[i] = dc(np.flip(liXs_np[i], axis=1))
        liXts[i] = dc(np.flip(liXts_np[i], axis=1))

    # Reshaping For Torch
    for i in range(len(liXs)):
        liXs_np[i] = liXs_np[i].reshape((-1, n_step, 1))
        liXts_np[i] = liXts_np[i].reshape((-1, n_step, 1))
        ys_np[i] = ys_np[i].reshape((-1, 1))
        yts_np[i] = yts_np[i].reshape((-1, 1))

    # To TensorTorch
    for i in range(len(liXs)):
        liXs_np[i] = torch.tensor(liXs_np[i]).float()
        liXts_np[i] = torch.tensor(liXts_np[i]).float()
        ys_np[i] = torch.tensor(ys_np[i]).float()
        yts_np[i] = torch.tensor(yts_np[i]).float()

    #print("Train Length " ,len(liXs_np[0]))
    #print("Test Length " ,len(liXts_np[0]))

    Xs_cls = []
    Xts_cls = []
    for i in range(len(liXs_np)):
        Xs_cls.append(TimeData(liXs_np[i], ys_np[i]))
        Xts_cls.append(TimeData(liXts_np[i], yts_np[i]))

    trn_load_vmd = []
    tst_load_vmd = []

    for i in range(len(Xs_cls)):
        trn_load_vmd.append(DataLoader(Xs_cls[i], batch_size=bch, shuffle=True))
        tst_load_vmd.append(DataLoader(Xts_cls[i], batch_size=bch, shuffle=True))

    return trn_load_vmd, tst_load_vmd

In [ ]:
# @title Preparing and Traning Stuffs

# Inserting VMD Tensors to Dataframe
def list2Df(df, li, col):
    li = li.tolist()
    dft = pd.DataFrame({col:li})
    return dft

# Preparing Steps In Dataframe
def preparing(df, step, textCol):
  df = dc(df)

  for i in range(1, step + 1):
    df[f'feat(t-{i})'] = df[textCol].shift(i)

  return df

# Class For New DataType
class TimeData(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, i):
    return self.X[i], self.y[i]

# For Converting to Tensor
def toTensor(data, op, show=False):
    if show:
        print(data)

    if op == 0:
        return torch.tensor(data).float()
    if op == 1:
        # Ensure data is converted to a torch.float32 tensor
        return torch.tensor(data, dtype=torch.float32)
    if op == 2:
        # Ensure data is converted to a torch.float32 tensor
        return torch.tensor(data, dtype=torch.float32)

# For Stacking Features
def stacker(data):
    data_tup = (toTensor(data[data.columns.values[0]], 0),)
    for i, dt in enumerate(data.columns.values):
        if i == 0:
            continue
        data_tup = data_tup + (toTensor(data[dt], 0),)
    return data_tup

# For Model Training And Evaluation
def validate_one_epoch(model, test, lossf_def, show_test, scalers):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model.train(False)
  running_loss = 0.0

  out_nov = []
  y_fsum = []

  for batch_index, batch in enumerate(test):
    x_batch, y_batch = batch[0].to(device), batch[1].to(device) # Move to device

    with torch.no_grad():
      # print("x_batch\n",x_batch.shape, x_batch)
      output = model(x_batch)

      loss = torch.sqrt(lossf_def(output, y_batch))
      running_loss += loss.item()

      # Convert torch.Tensor to numpy array for scaler, then inverse transform, then convert back to torch.Tensor
      # Explicitly reshape to (-1, 1) to ensure 2D input for inverse_transform
      output = scalers.inverse_transform(output.cpu().numpy())
      y_batch = scalers.inverse_transform(y_batch.cpu().numpy())

      # Append as torch.Tensor to out_nov and y_fsum, as expected by processResult
      out_nov.append(torch.from_numpy(output).to(device))
      y_fsum.append(torch.from_numpy(y_batch).to(device))

  avg_loss_across_batches = running_loss / len(test)

  if show_test:
    print('RMSE Loss: ',avg_loss_across_batches)
    print('---------------------------------------')

  return out_nov, avg_loss_across_batches, y_fsum

def train_one_epoch(model, train, optimz, lossf_def, epoch, max_norm, linear):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  torch.autograd.set_detect_anomaly(True)
  model.train(True)
  running_loss = 0.0

  for batch_index, batch in enumerate(train):
    # print(batch_index, batch)
    x_batch, y_batch = batch[0].to(device), batch[1].to(device)
    # x_batch, y_batch = batch[0], batch[1]

    output = model(x_batch)

    if output.isnan().any().item() or y_batch.isnan().any().item():
        print(output,' Those are the outputs, and these are the inputs ', x_batch)
        return True

    if linear:
        loss = torch.sqrt(lossf_def(output, y_batch))+0.5*torch.norm(torch.cat([torch.randn(n_steps, 1).view(-1)]), 1)
    else:
        loss = torch.sqrt(lossf_def(output, y_batch))
    running_loss += loss.item()

    optimz.zero_grad()
    loss.backward()

    #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

    optimz.step()

  return False

def train_test_v1(model, train, test, scalers, lrn=0.001, iterasi=50, loss=nn.MSELoss(), max_norm=1, show_test=False, linear=False):
  optimz = torch.optim.Adam(model.parameters(), lr=lrn)
  skipTrain = False

  for epoch in range(iterasi):
    if skipTrain == False:
        skipTrain = train_one_epoch(model, train, optimz, loss, epoch, max_norm, linear)
    out, rmse, y_fs = validate_one_epoch(model, test, loss, show_test, scalers)

  return out, rmse, y_fs

In [7]:
# @title Process Data For Non VMD
def processData(tr,ts, batches, n_step,colName):
  f_tr = torch.stack(stacker(tr), dim=1)
  f_ts = torch.stack(stacker(ts), dim=1)

  Xs = dc(tr)
  Xts = dc(ts)

  # These lines were the direct cause of the TypeError.
  # They attempt to re-assign the 'colName' column using `list2Df` which
  # converts the `(N,1)` tensor `f_tr` into a list of lists `[[f1],[f2],...]`
  # and then creates a DataFrame column of `dtype=object`.
  # Removing them as `Xs` and `Xts` are already correct `float` DataFrames
  # for the `preparing` function.
  # Xs = list2Df(Xs, f_tr, colName)
  # Xts = list2Df(Xts, f_ts, colName)

  X = preparing(Xs, n_step, colName)
  Xt = preparing(Xts, n_step, colName)

  X = X.drop(index=X.index[:n_step])
  Xt = Xt.drop(index=Xt.index[:n_step])

  # Splitting
  y = X.pop(colName)
  yt = Xt.pop(colName)

  # To Numpy
  X_np = X.to_numpy()
  Xt_np = Xt.to_numpy()

  # Flipping
  X_np = dc(np.flip(X_np, axis=1))
  Xt_np = dc(np.flip(Xt_np, axis=1))

  # Reshaping
  X_np = X_np.reshape((-1, n_steps, 1))
  Xt_np = Xt_np.reshape((-1, n_steps, 1))

  y_np = y.to_numpy().reshape((-1, 1))
  yt_np = yt.to_numpy().reshape((-1, 1))

  # To Tensor
  X_np = toTensor(X_np, 1)
  Xt_np = toTensor(Xt_np, 1)

  y_np = toTensor(y_np, 2)
  yt_np = toTensor(yt_np, 2)

  # To Timedata
  X_cls = TimeData(X_np, y_np)
  Xt_cls = TimeData(Xt_np, yt_np)

  trn_load = DataLoader(X_cls, batch_size=batches, shuffle=True)
  tst_load = DataLoader(Xt_cls, batch_size=batches, shuffle=True)

  return trn_load, tst_load, X_np, y_np, Xt_np, yt_np

In [8]:
# @title Process Result Function
def processResult(out, y_f, y_forsum, sums):
    for i, data in enumerate(out[0]):
        sums.append([])
        for a, dataA in enumerate(out):
            sums[i].append([])

    for i, data in enumerate(y_f[0]):
        y_forsum.append([])
        for a, dataA in enumerate(y_f):
            y_forsum[i].append([])

    #print(sum)
    for i, data in enumerate(out[0]):
        for a, dataA in enumerate(out):
            sums[i][a] = np.array(out[a][i].cpu())
        sums[i] = np.array(sums[i])
        sums[i] = sums[i].sum(axis=0)

    for i, data in enumerate(y_f[0]):
        for a, dataA in enumerate(y_f):
            y_forsum[i][a] = y_f[a][i].tolist()
        #print(y_forsum[i])
        y_forsum[i] = np.array(y_forsum[i])
        y_forsum[i] = y_forsum[i].sum(axis=0)

    for i, d in enumerate(sums):
        sums[i] = d.tolist()
        y_forsum[i] = y_forsum[i].tolist()

    out1 = []
    y1 = []
    #print(len(sum))
    #print(len(y_fsm))
    for i, d in enumerate(sums[0]):
        for o, g in enumerate(sums[0][i]):
            out1.append(g)
            y1.append(y_forsum[i][o])

    #print(sums, '\n')
    #print(y_forsum, '\n')
    #print(len(y1), '\n', y1, '\n\n', len(out1), '\n', out1)
    rmse_final = [math.sqrt(mean_squared_error(y1, out1)), mean_absolute_error(y1, out1)]
    print(rmse_final)
    return rmse_final

In [9]:
# @title Process Result & Train Model (No VMD)
def processResultNoVMD(out, y_f, y_forsum, sums):
    for i, data in enumerate(out[0]):
        sums.append([])
        for a, dataA in enumerate(out):
            sums[i].append([])

    for i, data in enumerate(y_f[0]):
        y_forsum.append([])
        for a, dataA in enumerate(y_f):
            y_forsum[i].append([])

    #print(sum)
    for i, data in enumerate(out[0]):
        for a, dataA in enumerate(out):
            sums[i][a] = np.array(out[a][i].cpu())
            # sums[i][a] = np.array(out[a][i])
        sums[i] = np.array(sums[i])
        sums[i] = sums[i].sum(axis=0)

    for i, data in enumerate(y_f[0]):
        for a, dataA in enumerate(y_f):
            y_forsum[i][a] = y_f[a][i].tolist()
        #print(y_forsum[i])
        y_forsum[i] = np.array(y_forsum[i])
        y_forsum[i] = y_forsum[i].sum(axis=0)

    for i, d in enumerate(sums):
        sums[i] = d.tolist()
        y_forsum[i] = y_forsum[i].tolist()

    out1 = []
    y1 = []
    #print(len(sum))
    #print(len(y_fsm))
    for i, d in enumerate(sums[0]):
        for o, g in enumerate(sums[0][i]):
            out1.append(g)
            y1.append(y_forsum[i][o])

    #print(sums, '\n')
    #print(y_forsum, '\n')
    # print(len(y1), '\n', y1, '\n\n', len(out1), '\n', out1)
    rmse_final = [math.sqrt(mean_squared_error(y1, out1)), mean_absolute_error(y1, out1)]
    print(rmse_final)
    return rmse_final

def trainModelNoVMD(train, test, epoch, model, scale):
  out = []
  y_fsm = []
  rmse_fin = []
  rmse = 0

  tmp, rmse_tmp, y_f = train_test_v1(model, train, test, scalers=scale, show_test=False, iterasi=epoch, linear=True)
  out.append(tmp)
  y_fsm.append(y_f)
  rmse_fin.append(rmse_tmp)
  rmse += rmse_tmp

  sumi = []
  y_forsum = []
  rmse = processResultNoVMD(out, y_fsm, y_forsum, sumi)

  return rmse

In [10]:
# @title Predict with VMD Function
def PredictVMD(tr, ts, batches, n_step, colName, scale):
    trnLSTM, tstLSTM = TheVMD(tr, ts, colName, 5000, 0, 30, 1e-5, batches, n_step)
    trnBLSTM, tstBLSTM = TheVMD(tr, ts, colName, 7000, 1, 20, 1e-7, batches, n_step)
    trnRNN, tstRNN = TheVMD(tr, ts, colName, 7000, 0, 30, 1e-5, batches, n_step)
    trnLin, tstLin = TheVMD(tr, ts, colName, 9000, 0, 10, 1e-7, batches, n_step)
    trnDLin, tstDLin = TheVMD(tr, ts, colName, 9000, 0, 30, 1e-5, batches, n_step)
    trnNLin, tstNLin = TheVMD(tr, ts, colName, 3000, 0, 20, 1e-7, batches, n_step)

    li_lstm = []
    li_blstm = []
    li_rnn = []
    li_lin = []
    li_dlin = []
    li_nlin = []

    # Define The Models
    for modi in range(len(trnLin)):
        lin = Lin(n_step, batches, 1, individual=True) # Dont Forget To Set Epoch = 30
        lin.to(device)
        li_lin.append(lin)

    for magni in range(len(trnDLin)):
        dlin = DLin(n_step, batches, 1, individual=True) # Dont Forget To Set Epoch = 60
        dlin.to(device)
        li_dlin.append(dlin)

        lstm = LSTM(1, 30, 1) # Dont Forget To Set Epoch = 70
        lstm.to(device)
        li_lstm.append(lstm)

        rnn = SimpRNN(1, 40, 1) # Dont Forget To Set Epoch = 30
        rnn.to(device)
        li_rnn.append(rnn)

    for lavender in range(len(trnNLin)):
        nlin = NLin(n_step, batches, 1, individual=True) # Dont Forget To Set Epoch = 70
        nlin.to(device)
        li_nlin.append(nlin)

        blstm = BLSTM(1, 10, 1) # Dont Forget To Set Epoch = 30
        blstm.to(device)
        li_blstm.append(blstm)

    # Start Predicting
    outLin = []
    y_fsmLin = []
    rmse_finLin = []
    rmseLin = 0

    outDLin = []
    y_fsmDLin = []
    rmse_finDLin = []
    rmseDLin = 0

    outNLin = []
    y_fsmNLin = []
    rmse_finNLin = []
    rmseNLin = 0

    out = []
    y_fsm = []
    rmse_fin = []
    rmse = 0

    outBLSTM = []
    y_fsmBLSTM = []
    rmse_finBLSTM = []
    rmseBLSTM = 0

    outRNN = []
    y_fsmRNN = []
    rmse_finRNN = []
    rmseRNN = 0

    for i in range(len(li_lstm)):
        tmp, rmse_tmp, y_f = train_test_v1(li_lstm[i], trnLSTM[i], tstLSTM[i], scalers=scale, show_test=False, iterasi=70, linear=True)
        out.append(tmp)
        y_fsm.append(y_f)
        rmse_fin.append(rmse_tmp)
        rmse += rmse_tmp

        tmp, rmse_tmp, y_f = train_test_v1(li_rnn[i], trnRNN[i], tstRNN[i], scalers=scale, show_test=False, iterasi=30, linear=True)
        outRNN.append(tmp)
        y_fsmRNN.append(y_f)
        rmse_finRNN.append(rmse_tmp)
        rmseRNN += rmse_tmp

        tmp, rmse_tmp, y_f = train_test_v1(li_dlin[i], trnDLin[i], tstDLin[i], scalers=scale, show_test=False, iterasi=60, linear=True)
        outDLin.append(tmp)
        y_fsmDLin.append(y_f)
        rmse_finDLin.append(rmse_tmp)
        rmseDLin += rmse_tmp

    print("DLin, LSTM, RNN Done")
    for a in range(len(li_blstm)):
        tmp, rmse_tmp, y_f = train_test_v1(li_nlin[a], trnNLin[a], tstNLin[a], scalers=scale, show_test=False, iterasi=70, linear=True)
        outNLin.append(tmp)
        y_fsmNLin.append(y_f)
        rmse_finNLin.append(rmse_tmp)
        rmseNLin += rmse_tmp

        tmp, rmse_tmp, y_f = train_test_v1(li_blstm[a], trnBLSTM[a], tstBLSTM[a], scalers=scale, show_test=False, iterasi=30, linear=True)
        outBLSTM.append(tmp)
        y_fsmBLSTM.append(y_f)
        rmse_finBLSTM.append(rmse_tmp)
        rmseBLSTM += rmse_tmp

    print("NLin & BLSTM Done")
    for o in range(len(li_lin)):
        tmp, rmse_tmp, y_f = train_test_v1(li_lin[o], trnLin[o], tstLin[o], scalers=scale, show_test=False, iterasi=30, linear=True)
        outLin.append(tmp)
        y_fsmLin.append(y_f)
        rmse_finLin.append(rmse_tmp)
        rmseLin += rmse_tmp

    print("Lin Done")
    # Process The Result For RNN Based
    sumi = []
    y_forsum = []
    rmseLSTM = processResult(out, y_fsm, y_forsum, sumi)

    sumBLSTM = []
    y_forsumBLSTM = []
    rmseBLSTM = processResult(outBLSTM, y_fsmBLSTM, y_forsumBLSTM, sumBLSTM)

    sumRNN = []
    y_forsumRNN = []
    rmseRNN = processResult(outRNN, y_fsmRNN, y_forsumRNN, sumRNN)

    # Process The Result For Linear Based
    sumLin = []
    y_forsumLin = []
    rmseLin = processResult(outLin, y_fsmLin, y_forsumLin, sumLin)

    sumDLin = []
    y_forsumDLin = []
    rmseDLin = processResult(outDLin, y_fsmDLin, y_forsumDLin, sumDLin)

    sumNLin = []
    y_forsumNLin = []
    rmseNLin = processResult(outNLin, y_fsmNLin, y_forsumNLin, sumNLin)

    return [rmseLSTM, rmseBLSTM, rmseRNN, rmseLin, rmseDLin, rmseNLin]

In [11]:
# @title Predict No VMD Fucntion
def PredictNoVMDLin(tr, ts, batches, n_step, colName, scale):
  trn, tst, x_np, y_np, xt_np, yt_np = processData(tr, ts, batches, n_step, colName)

  lin = Lin(n_step, 1, 1, individual=True) # Dont Forget To Set Epoch = 30
  lin.to(device)

  dlin = DLin(n_step, 1, 1, individual=True) # Dont Forget To Set Epoch = 60
  dlin.to(device)

  nlin = NLin(n_step, 1, 1, individual=True) # Dont Forget To Set Epoch = 70
  nlin.to(device)

  rmseLin = trainModelNoVMD(trn, tst, 30, lin, scale)
  rmseDLin = trainModelNoVMD(trn, tst, 60, dlin, scale)
  rmseNLin = trainModelNoVMD(trn, tst, 70, nlin, scale)

  return [rmseLin, rmseDLin, rmseNLin]

def PredictNoVMD(tr, ts, batches, n_step, colName, scale):
  trn, tst, x_np, y_np, xt_np, yt_np = processData(tr, ts, batches, n_step, colName)

  lstm = LSTM(1, 30, 1) # Dont Forget To Set Epoch = 70
  lstm.to(device)

  blstm = BLSTM(1, 10, 1) # Dont Forget To Set Epoch = 30
  blstm.to(device)

  rnn = SimpRNN(1, 40, 1) # Dont Forget To Set Epoch = 30
  rnn.to(device)

  rmseLSTM = trainModelNoVMD(trn, tst, 70, lstm, scale)
  rmseBLSTM = trainModelNoVMD(trn, tst, 30, blstm, scale)
  rmseRNN = trainModelNoVMD(trn, tst, 30, rnn, scale)

  return [rmseLSTM, rmseBLSTM, rmseRNN]

In [ ]:
rmseModel = []

for idx, data in enumerate(trn_li):
    print("Data ke-", idx)
    col = list(data.columns.values)[-1]
    trn_li[idx] = pd.DataFrame(trn_li[idx][col])
    tst_li[idx] = pd.DataFrame(tst_li[idx][col])

    tmpModel = PredictVMD(trn_li[idx], tst_li[idx], 6, 6, col, scale_li[idx])
    print(tmpModel)
    print("VMD RMSE LSTM, BLSTM, RNN, Lin, DLin, NLin: ", tmpModel)
    rmseModel.append(tmpModel)

with open("/Result/resultUniVMD.pkl", 'wb') as f:
    pickle.dump(rmseModel, f)
    print("File Successfully Created")

In [ ]:
rmseModelNoVMD = []

for idx, data in enumerate(trn_li):
    print("Data ke-", idx)
    col = list(data.columns.values)[-1]
    trn_li[idx] = pd.DataFrame(trn_li[idx][col])
    tst_li[idx] = pd.DataFrame(tst_li[idx][col])

    tmpModel = PredictNoVMD(trn_li[idx], tst_li[idx], 6, 6, col, scale_li[idx])
    print("RMSE LSTM, BLSTM, RNN: ", tmpModel)
    tmpModel1 = PredictNoVMDLin(trn_li[idx], tst_li[idx], 6, 6, col, scale_li[idx])
    print("RMSE Lin, DLin, NLin: ", tmpModel1)
    rmseModelNoVMD.append(tmpModel + tmpModel1)

with open("/Result/resultUni.pkl", 'wb') as f:
    pickle.dump(rmseModelNoVMD, f)
    print("File Successfully Created")